## Documentacion oficial de PyTorch

- `torch.nn.RNN`: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html

## Objetivo del notebook

La idea de este bloque es cambiar hiperparámetros y dimensiones de entrada para observar cómo cambian:
- los tensores de entrada y salida,
- el estado oculto final,
- los parámetros entrenables del modelo.

En todos los ejemplos vamos a usar la convencion de PyTorch con `batch_first=True`:

- `x`: `[batch_size, largo_secuencia, input_size]`
- `output`: `[batch_size, largo_secuencia, hidden_size]`
- `h_n`: `[num_layers, batch_size, hidden_size]`


### Funciones comunes a usar

In [ ]:
# Librerias
import torch


In [ ]:
# Clase minima para inspeccionar una RNN basica
class SimpleRNN(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=1, num_layers=1):
        super().__init__()
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, x):
        output, h_n = self.rnn(x)
        return output, h_n


In [ ]:
# Esta función muestra los parámetros entrenables y sus dimensiones.

def imp_param(model):
    print('-' * 84)
    print('PARAMETROS DEL MODELO')
    print('-' * 84)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f'{name}: {tuple(param.shape)}')
            total += param.numel()
    print()
    print(f'Total de parámetros entrenables: {total}')


In [ ]:
# Esta función imprime de forma ordenada la salida de un modelo.
#
# La usamos porque, según el modelo, la salida puede venir en distintos formatos:
# - un tensor único,
# - una tupla o lista de tensores,
# - o incluso un diccionario.
#
# En RNN simples de PyTorch, por ejemplo, el forward suele devolver:
#   output, h_n
# o sea, una tupla con dos tensores.
#
# La idea es recorrer recursivamente esa estructura y mostrar, para cada elemento,
# su nombre y su shape, sin asumir de antemano cómo viene empaquetada la salida.
def mostrar_tensores(obj, nombre='salida'):
    # Caso 1: el objeto ya es un tensor. Mostramos su shape y su contenido.
    if isinstance(obj, torch.Tensor):
        print(f'{nombre}: shape = {tuple(obj.shape)}')
        print(obj)
    # Caso 2: el objeto es una tupla o lista. Recorremos cada posición.
    elif isinstance(obj, (tuple, list)):
        for i, item in enumerate(obj):
            mostrar_tensores(item, nombre=f'{nombre}[{i}]')
    # Caso 3: el objeto es un diccionario. Recorremos cada clave.
    elif isinstance(obj, dict):
        for key, value in obj.items():
            mostrar_tensores(value, nombre=f'{nombre}["{key}"]')
    # Caso 4: cualquier otro tipo. Lo mostramos para detectar salidas no esperadas.
    else:
        print(f'{nombre}: {type(obj)}')
        print(obj)


# Esta función arma una entrada aleatoria, ejecuta un forward y resume las shapes principales.
def teoria(model, largo_entrada=3, batch_size=1, input_size=1):
    print('-' * 84)
    print('MODELO')
    print('-' * 84)
    print(model)
    imp_param(model)

    entrada = torch.rand(batch_size, largo_entrada, input_size)
    print('-' * 84)
    print('ENTRADA')
    print('-' * 84)
    print(f'entrada shape: {tuple(entrada.shape)}')
    print(entrada)

    salida = model(entrada)
    print('-' * 84)
    print('SALIDA')
    print('-' * 84)
    mostrar_tensores(salida)

    return entrada, salida


### Ejemplo guiado 1 - RNN simple

![RNN basica](rnn_basica.png)

Complete las dimensiones de las siguientes variables antes de ejecutar la celda:

```
# Entrada
x =

# Salidas
output =
h_n =

# Parametros
weight_ih_l0 =
weight_hh_l0 =
bias_ih_l0 =
bias_hh_l0 =
```


In [ ]:
# Primer ejemplo: RNN con una sola feature, un hidden y una sola capa.

input_size = 1
hidden_size = 1
num_layers = 1
largo_entrada = 7
batch_size = 1

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


### Ejemplo guiado 2 - Hidden mayor a 1

![RNN con hidden](rnn_hidden.png)

Prediga como cambian las dimensiones al aumentar `hidden_size`.

```
# Entrada
x =

# Salidas
output =
h_n =

# Parametros
weight_ih_l0 =
weight_hh_l0 =
```


In [ ]:
# Segundo ejemplo: mantenemos la entrada simple pero aumentamos el hidden_size.

input_size = 1
hidden_size = 3
num_layers = 1
largo_entrada = 7
batch_size = 1

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


### Ejemplo guiado 3 - Input multivariable

![RNN multivariable](rnn_multivariable.png)

Ahora la secuencia tiene varias features por instante.

```
# Entrada
x =

# Salidas
output =
h_n =

# Parametros
weight_ih_l0 =
weight_hh_l0 =
```


In [ ]:
# Tercer ejemplo: ahora cada instante temporal tiene varias features de entrada.

input_size = 3
hidden_size = 2
num_layers = 1
largo_entrada = 7
batch_size = 1

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


### Ejemplo guiado 4 - Multilayer y batch

![RNN por capas](rnn_layer.png)

Observe como cambia `h_n` cuando hay varias capas y varias muestras por batch.

```
# Entrada
x =

# Salidas
output =
h_n =

# Parametros
weight_ih_l0 =
weight_hh_l0 =
weight_ih_l1 =
weight_hh_l1 =
```


In [ ]:
# Cuarto ejemplo: combinamos varias capas y varias muestras por batch.

input_size = 2
hidden_size = 3
num_layers = 2
largo_entrada = 9
batch_size = 4

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


### Mini experimento

Para una `RNN` simple de una sola capa, compare `output[:, -1, :]` con `h_n[-1]`.

Pregunta: ¿representan la misma información?


In [ ]:
# Verificamos numéricamente la relación entre el último output temporal y h_n[-1].

input_size = 2
hidden_size = 4
num_layers = 1
largo_entrada = 5
batch_size = 3

model = SimpleRNN(input_size, hidden_size, num_layers)
entrada = torch.rand(batch_size, largo_entrada, input_size)
output, h_n = model(entrada)

print('output[:, -1, :].shape =', output[:, -1, :].shape)
print('h_n[-1].shape =', h_n[-1].shape)
print('¿Son iguales? ->', torch.allclose(output[:, -1, :], h_n[-1]))


## Ejercicio 1 - Complete el cuadro del powerpoint para los siguientes ejemplos

Ejecute cada caso y complete manualmente las dimensiones de entradas, salidas y parámetros.


```
EJEMPLO A
input_size = 2
batch_size = 16
hidden_size = 24
num_layers = 3
```


In [ ]:
# Caso A del ejercicio 1: usamos estos hiperparámetros para completar las dimensiones.

input_size = COMPLETAR
batch_size = COMPLETAR
hidden_size = COMPLETAR
num_layers = COMPLETAR
largo_entrada = COMPLETAR

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


```
EJEMPLO B
input_size = 6
batch_size = 16
hidden_size = 24
num_layers = 1
```


In [ ]:
# Caso B del ejercicio 1: mismo procedimiento, con otra cantidad de features.

input_size = COMPLETAR
batch_size = COMPLETAR
hidden_size = COMPLETAR
num_layers = COMPLETAR
largo_entrada = COMPLETAR

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


```
EJEMPLO C
input_size = 10
batch_size = 64
hidden_size = 64
num_layers = 4
```


In [ ]:
# Caso C del ejercicio 1: ejemplo más grande para practicar lectura de shapes.

input_size = COMPLETAR
batch_size = COMPLETAR
hidden_size = COMPLETAR
num_layers = COMPLETAR
largo_entrada = COMPLETAR

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


## Ejercicio 2 - Implementar una RNN para clasificacion de 5 clases

Implemente una red con:
- `input_size = 2`
- `hidden_size = 40`
- `num_layers = 2`
- una capa `Linear` final con `out_features = 5`

Condiciones:
- usar el ultimo estado temporal de `output` como entrada a la `Linear`,
- devolver `logits` y no probabilidades,
- probar el funcionamiento del modelo con una entrada aleatoria sin entrenar la red.

![RNN clasificacion](rnn_class.png)


In [ ]:
# Escriba aqui su solucion
# Sugerencia: el forward puede devolver un tensor de shape [batch_size, n_clases]

class RNNClasificacion(torch.nn.Module):
    def __init__(self, input_size=2, hidden_size=40, num_layers=2, n_clases=5):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        raise NotImplementedError('Completar como parte del ejercicio')
